Install the dependencies from `requirements.txt` before running this notebook.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import getpass

In [ ]:
password = getpass.getpass()

In [ ]:
db = "sakila"
connection_string = 'mysql+pymysql://root:' + password + '@localhost/' + db
engine = create_engine(connection_string)

In [ ]:
def rentals_month(engine, month, year):
    query = text("""
        SELECT * FROM rental
        WHERE MONTH(rental_date) = :month
          AND YEAR(rental_date) = :year
    """)
    with engine.connect() as connection:
        result = connection.execute(query, {"month": month, "year": year})
        df = pd.DataFrame(result.all())
    return df

In [ ]:
may_df = rentals_month(engine, 5, 2005)
may_df.head()

In [ ]:
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"
    df_grouped = df.groupby('customer_id').size().reset_index(name=column_name)
    return df_grouped

In [ ]:
may_count = rental_count_month(may_df, 5, 2005)
may_count.head()

In [ ]:
def compare_rentals(df1, df2):
    """Compare rental counts for customers active in both months."""
    merged = pd.merge(df1, df2, on="customer_id", how="inner")
    count_columns = [column for column in merged.columns if column != "customer_id"]
    merged["difference"] = merged[count_columns[1]] - merged[count_columns[0]]
    return merged


In [ ]:
june_df = rentals_month(engine, 6, 2005)
june_count = rental_count_month(june_df, 6, 2005)

comparison = compare_rentals(may_count, june_count)
comparison.head()